# 2. Decomposition

Classical additive/multiplicative and robust STL. Owner: Prem — Phase 2.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))
import os
os.chdir(project_root)

from src.decomposition import residual_summary, run_all
from src.eda import impute_linear, suggest_sarima_orders, adf_table
from src.load_data import load_daily_series
from src.plots import plot_decompose, save_current_figure

DATA_PATH = project_root / "data/processed/pm25_paris_daily.csv"
(project_root / "reports/figures").mkdir(parents=True, exist_ok=True)

series = impute_linear(load_daily_series(DATA_PATH))
series.head()

date
2024-01-01    17.000000
2024-01-02    19.355198
2024-01-03    19.943599
2024-01-04    18.328985
2024-01-05    15.733609
Freq: D, Name: pm25, dtype: float64

## Classical seasonal decomposition

Period = 7 (weekly). Additive assumes constant seasonal amplitude; multiplicative scales with the level.

In [2]:
PERIOD = 7
bundle = run_all(series, period=PERIOD)

plot_decompose(bundle.additive, "Classical additive (period=7)")
save_current_figure("02_classical_additive.png")
plt.show()

plot_decompose(bundle.multiplicative, "Classical multiplicative (period=7)")
save_current_figure("02_classical_multiplicative.png")
plt.show()

/var/folders/8n/mwqyfv8j437btzxfh5c852n00000gn/T/ipykernel_23226/267949119.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/8n/mwqyfv8j437btzxfh5c852n00000gn/T/ipykernel_23226/267949119.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Robust STL decomposition

In [3]:
plot_decompose(bundle.stl, "STL robust=True (period=7)")
save_current_figure("02_stl_robust.png")
plt.show()

summary = pd.DataFrame(
    [
        residual_summary(bundle.additive, "classical_additive"),
        residual_summary(bundle.multiplicative, "classical_multiplicative"),
        residual_summary(bundle.stl, "stl_robust"),
    ]
)
summary

/var/folders/8n/mwqyfv8j437btzxfh5c852n00000gn/T/ipykernel_23226/2659663968.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,method,resid_std,resid_mean,seasonal_amp
0,classical_additive,0.055618,-0.000107,5.849187
1,classical_multiplicative,0.008559,0.999894,0.371532
2,stl_robust,0.060850,-0.004452,5.849745


## Interpretation and suggested SARIMA orders

### Trend
The trend component captures the slow level of Paris PM2.5 (typically higher in colder months). Classical and STL agree on the broad shape; STL is smoother because LOESS down-weights short spikes.

### Seasonality
With `period=7` both methods show a clear weekly cycle (traffic / weekday activity). Multiplicative seasonality stretches when the level is high; additive keeps a fixed amplitude. For this series additive + STL are usually enough; multiplicative is kept as a comparison.

### Residuals
Residuals hold irregular pollution episodes and sensor noise. STL (`robust=True`) pushes more of those spikes into the residual than classical, which is useful later for Vishal’s anomaly detection on STL residuals.

### Classical vs STL
- Classical is fast and simple but sensitive to outliers.
- STL is more flexible (LOESS trend/seasonal) and robust mode limits outlier influence on the seasonal pattern.

### Orders for Yahya
See also `docs/sarima_orders_for_yahya.md`.

In [4]:
adf = adf_table(series)
prefer_diff = not bool(adf.loc[adf["series"] == "raw", "stationary_5pct"].iloc[0])
orders = suggest_sarima_orders(seasonal_period=PERIOD, prefer_diff=prefer_diff)
print("ADF table:")
display(adf)
print("Suggested SARIMA order:", orders["order"])
print("Suggested seasonal_order:", orders["seasonal_order"])
print("Message Yahya with these starters so he can begin Phase 3.")

ADF table:


/Users/prem/tsa-group5-air-quality/src/eda.py:56: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, pvalue, used_lag, nobs, *_ = adfuller(clean, autolag="AIC")
/Users/prem/tsa-group5-air-quality/src/eda.py:56: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, pvalue, used_lag, nobs, *_ = adfuller(clean, autolag="AIC")


,series,adf_stat,pvalue,used_lag,nobs,stationary_5pct
0,raw,-2.154243,0.223269,15,194,False
1,diff_1,-0.581568,0.875093,15,193,False


Suggested SARIMA order: (1, 1, 1)
Suggested seasonal_order: (1, 1, 1, 7)
Message Yahya with these starters so he can begin Phase 3.
